In [0]:
import os, sys, time
REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from datetime import datetime, timezone
from pyspark.sql import functions as F, Row
from src.transforms.vehicle_positions import (VP_KEY, CARRIAGE_KEY, flatten_vehicle_positions,
                                              flatten_carriages, collapse_to_grain)

spark.conf.set("spark.sql.session.timeZone", "UTC")
TZ         = "America/New_York"
BRONZE     = "transit.bronze.rt_vehicle_positions"
SILVER_VP  = "transit.silver.vehicle_positions"
SILVER_CAR = "transit.silver.vehicle_carriages"
PIPE       = "silver_vehicle_positions"
LOOKBACK_S = 3600       # re-read the last hour on every run

print("ready")

In [0]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS transit.ops.watermarks (
    pipeline STRING, watermark BIGINT, rows_read BIGINT,
    rows_inserted BIGINT, rows_updated BIGINT, updated_at TIMESTAMP)
""")

def get_watermark(pipeline):
    r = (spark.table("transit.ops.watermarks")
           .filter(F.col("pipeline") == pipeline).select("watermark").first())
    return int(r[0]) if r else 0

def set_watermark(pipeline, value, rows_read=0, inserted=0, updated=0):
    spark.sql(f"""
      MERGE INTO transit.ops.watermarks t
      USING (SELECT '{pipeline}' AS pipeline,
                    CAST({int(value)} AS BIGINT)     AS watermark,
                    CAST({int(rows_read)} AS BIGINT) AS rows_read,
                    CAST({int(inserted)} AS BIGINT)  AS rows_inserted,
                    CAST({int(updated)} AS BIGINT)   AS rows_updated,
                    current_timestamp()              AS updated_at) s
      ON t.pipeline = s.pipeline
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)

def log_run(**fields):
    (spark.createDataFrame([Row(run_at=datetime.now(timezone.utc), **fields)])
       .write.format("delta").mode("append").option("mergeSchema", "true")
       .saveAsTable("transit.ops.pipeline_runs"))

print("watermark now:", get_watermark(PIPE))

In [0]:
flat_all = flatten_vehicle_positions(spark.table(BRONZE), TZ)
worst = (flat_all.groupBy(*VP_KEY).count()
           .filter("count > 1").orderBy(F.desc("count")).first())
demo_src = flat_all.filter((F.col("vehicle_id") == worst.vehicle_id) &
                           (F.col("vehicle_ts") == worst.vehicle_ts))
print(f"vehicle {worst.vehicle_id} at {worst.vehicle_ts}: one observation, "
      f"present in {worst['count']} snapshots")

DEMO = "transit.silver._merge_demo"
spark.sql(f"DROP TABLE IF EXISTS {DEMO}")
demo_src.limit(0).write.format("delta").saveAsTable(DEMO)
demo_src.createOrReplaceTempView("demo_src")

naive_merge = f"""
  MERGE INTO {DEMO} t USING demo_src s
  ON t.vehicle_id = s.vehicle_id AND t.vehicle_ts = s.vehicle_ts
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *"""

spark.sql(naive_merge)
print(f"run 1, empty target: {spark.table(DEMO).count()} rows for ONE observation, and no error")

try:
    spark.sql(naive_merge)
    print("run 2: no error")
except Exception as e:
    print("run 2 FAILED:", str(e)[:300])

spark.sql(f"DROP TABLE {DEMO}")

In [0]:
def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()["version"]

def merge_observations(src, target, keys):
    """Upsert pre-collapsed observations into target. Returns {'inserted', 'updated'}."""
    src = src.withColumn("_silver_loaded_at", F.current_timestamp())
    src.limit(0).write.format("delta").mode("ignore").saveAsTable(target)   # create once
    before = table_version(target)
    src.createOrReplaceTempView("merge_src")

    on = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    newer = [c for c in src.columns if c not in keys and c != "first_seen_snapshot_ts"]
    set_newer = ", ".join(
        [f"t.`{c}` = s.`{c}`" for c in newer] +
        ["t.first_seen_snapshot_ts = least(t.first_seen_snapshot_ts, s.first_seen_snapshot_ts)"])

    spark.sql(f"""
      MERGE INTO {target} t
      USING merge_src s
      ON {on}
      WHEN MATCHED AND s.last_seen_snapshot_ts > t.last_seen_snapshot_ts
        THEN UPDATE SET {set_newer}
      WHEN MATCHED AND s.first_seen_snapshot_ts < t.first_seen_snapshot_ts
        THEN UPDATE SET t.first_seen_snapshot_ts = s.first_seen_snapshot_ts,
                        t._silver_loaded_at      = s._silver_loaded_at
      WHEN NOT MATCHED THEN INSERT *
    """)

    h = spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").first()
    if h["version"] == before:
        return {"inserted": 0, "updated": 0}
    m = h["operationMetrics"] or {}
    return {"inserted": int(m.get("numTargetRowsInserted", 0)),
            "updated":  int(m.get("numTargetRowsUpdated", 0))}

print("merge ready")

In [0]:
def run_incremental():
    wm = get_watermark(PIPE)
    lower = max(wm - LOOKBACK_S, 0)

    batch = (spark.table(BRONZE)
               .filter(F.col("_dt") >= F.date_sub(F.to_date(F.from_unixtime(F.lit(lower))), 1))  # prune
               .filter(F.col("_snapshot_ts") > lower))

    n_files = batch.select("_source_file").distinct().count()
    if n_files == 0:
        print("nothing new"); return
    batch_max = batch.agg(F.max("_snapshot_ts")).first()[0]

    flat = flatten_vehicle_positions(batch, TZ)
    bad = flat.filter(F.col("vehicle_id").isNull() | F.col("vehicle_ts").isNull()).count()
    assert bad == 0, f"{bad} rows with a null grain key - they would re-insert on every run"
    n_read = flat.count()
    vp = merge_observations(collapse_to_grain(flat, VP_KEY), SILVER_VP, VP_KEY)

    car = flatten_carriages(batch)
    bad = car.filter(" OR ".join(f"{k} IS NULL" for k in CARRIAGE_KEY)).count()
    assert bad == 0, f"{bad} carriage rows with a null grain key"
    cg = merge_observations(collapse_to_grain(car, CARRIAGE_KEY), SILVER_CAR, CARRIAGE_KEY)

    # only after both merges have committed
    set_watermark(PIPE, max(wm, batch_max), n_read, vp["inserted"], vp["updated"])
    log_run(pipeline=PIPE, watermark_before=wm, watermark_after=max(wm, batch_max),
            files_read=n_files, rows_read=n_read,
            vp_inserted=vp["inserted"], vp_updated=vp["updated"],
            car_inserted=cg["inserted"], car_updated=cg["updated"])

    print(f"read {n_files} snapshots / {n_read:,} rows since {lower} | "
          f"positions +{vp['inserted']:,} ~{vp['updated']:,} | "
          f"carriages +{cg['inserted']:,} ~{cg['updated']:,} | "
          f"watermark {wm} -> {max(wm, batch_max)}")

t0 = time.time()
run_incremental()
print(f"{time.time() - t0:.0f}s")

In [0]:
for i in range(3):
    run_incremental()
    print(f"  after run {i+2}: positions {spark.table(SILVER_VP).count():,} · "
          f"carriages {spark.table(SILVER_CAR).count():,}")

In [0]:
spark.sql(f"UPDATE transit.ops.watermarks SET watermark = 0 WHERE pipeline = '{PIPE}'")
print("watermark reset to 0 - re-reading all of bronze")
run_incremental()

b = spark.table(BRONZE)
expected_vp  = flatten_vehicle_positions(b, TZ).select(*VP_KEY).distinct().count()
expected_car = flatten_carriages(b).select(*CARRIAGE_KEY).distinct().count()
print(f"positions: silver {spark.table(SILVER_VP).count():,} vs distinct in bronze {expected_vp:,}")
print(f"carriages: silver {spark.table(SILVER_CAR).count():,} vs distinct in bronze {expected_car:,}")

In [0]:
vp = spark.table(SILVER_VP)
(vp.withColumn("fleet", F.substring("vehicle_id", 1, 1))
   .withColumn("age_at_capture_s", F.col("first_seen_snapshot_ts") - F.col("vehicle_ts"))
   .withColumn("frozen_h", (F.col("last_seen_snapshot_ts") - F.col("first_seen_snapshot_ts")) / 3600)
   .groupBy("fleet")
   .agg(F.count("*").alias("observations"),
        F.expr("percentile_approx(age_at_capture_s, 0.5)").alias("median_age_at_capture_s"),
        F.round(F.max("frozen_h"), 1).alias("longest_frozen_h"),
        F.sum((F.col("frozen_h") >= 1).cast("int")).alias("frozen_over_1h"))
   .orderBy(F.desc("observations")).display())

In [0]:
vp, cg = spark.table(SILVER_VP), spark.table(SILVER_CAR)
b = spark.table(BRONZE)

dup_vp  = vp.groupBy(*VP_KEY).count().filter("count > 1").count()
dup_car = cg.groupBy(*CARRIAGE_KEY).count().filter("count > 1").count()
orphans = cg.join(vp.select(*VP_KEY), VP_KEY, "left_anti").count()
wm, bmax = get_watermark(PIPE), b.agg(F.max("_snapshot_ts")).first()[0]

print(f"duplicate grain keys: positions {dup_vp}, carriages {dup_car}")
print(f"carriages with no parent position: {orphans}")
print(f"watermark {wm} vs newest bronze snapshot {bmax}")

assert vp.count() == expected_vp,  "positions don't match distinct bronze observations"
assert cg.count() == expected_car, "carriages don't match distinct bronze observations"
assert dup_vp == 0 and dup_car == 0, "duplicate grain keys"
assert orphans == 0, "carriages without a parent vehicle observation"
assert wm == bmax, "watermark is not at the newest bronze snapshot"
print("\n2.3 done")

spark.table("transit.ops.pipeline_runs").orderBy(F.desc("run_at")).limit(6).display()